In [2]:
# Gerekli Keras katmanlarını ve model yapısını içeri aktarıyoruz
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

# Sequential: Katmanları arka arkaya, sırayla (bir tünel gibi) dizeceğimizi belirtir.
model = Sequential()

# =====================================================================
# BÖLÜM 1: ÖZELLİK ÇIKARIMI (GÖZLER - CONVOLUTION & POOLING)
# =====================================================================

# --- 1. BLOK: Yüzeysel Özellikler (Düz çizgiler, kenarlar) ---
# Giriş: Hastaneden gelen tek bir orijinal resim (224x224 boyutunda, 3 renk kanallı).
# İşlem: 32 farklı 3x3 büyüteç, resmi tarayarak yatay/dikey çizgi haritaları çıkarır.
model.add(Conv2D(filters=32, kernel_size=(3, 3), activation='relu', input_shape=(224, 224, 3)))
# Küçültme: 2x2'lik çerçevelerle sadece en parlak/belirgin pikselleri alıp resmi yarı yarıya küçültür.
# Çıkış: Orijinal resim çöpe gider. Elimizde 112x112 boyutunda, 32 katmanlı (desteli) bir özellik haritası kalır.
model.add(MaxPooling2D(pool_size=(2, 2)))


# --- 2. BLOK: Orta Düzey Özellikler (Şekiller, kemik ve doku kıvrımları) ---
# İşlem: Buradaki 64 büyütecin her biri 3 boyutludur (3x3x32). Yani 1. bloktan gelen 32 katmanlı
# desteye zımba gibi yukarıdan aşağıya basar, hepsini harmanlayıp ezerek yepyeni tek bir resim çıkarır.
# Bu zımba işlemi 64 farklı büyüteçle yapıldığı için 64 yeni harita oluşur.
model.add(Conv2D(filters=64, kernel_size=(3, 3), activation='relu'))
# Küçültme: 112x112'lik haritaları tekrar yarı yarıya özetler.
# Çıkış: Elimizde 56x56 boyutunda, 64 katmanlı (desteli) daha soyut bir özellik haritası oluşur.
model.add(MaxPooling2D(pool_size=(2, 2)))


# --- 3. BLOK: Derin Analiz (Zatürreye özgü beyazlıklar, bulutsu lekeler) ---
# İşlem: 128 farklı 3 boyutlu büyüteç (3x3x64), 64'lü desteye basarak hastalık lekelerini parlatır.
model.add(Conv2D(filters=128, kernel_size=(3, 3), activation='relu'))
# Küçültme: 56x56'lık haritaları tekrar özetler.
# Çıkış: Elimizde 28x28 boyutunda, 128 katmanlı, sadece hastalıklı bölgelerin parladığı haritalar kalır.
model.add(MaxPooling2D(pool_size=(2, 2)))


# --- 4. BLOK: En İnce Detaylar (Son Damıtma) ---
# İşlem: 128 filtre (3x3x128), hastalık haritalarının üzerinden son kez geçip en soyut, en kritik izleri damıtır.
model.add(Conv2D(filters=128, kernel_size=(3, 3), activation='relu'))
# Küçültme: 28x28'lik haritaları son kez yarı yarıya küçültür.
# Çıkış (Karar Merkezine Gidecek Veri): Artık elimizde resim değil; 14x14 boyutunda, 128 katmanlı, 
# "Hastalık lekeleri tam olarak nerede ve ne yoğunlukta?" bilgisini taşıyan saf bir matematiksel rapor var.
model.add(MaxPooling2D(pool_size=(2, 2)))

c:\Users\hp\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Faz 3: Model Oluşturma (Model Construction) Adımları ve Mimari Tercihler

Veri ön işleme aşamasından geçen standartlaştırılmış (224x224x3 boyutlarındaki) röntgen görüntülerini analiz etmek, hastalık belirtilerini otonom olarak saptamak ve yüksek doğrulukla sınıflandırmak amacıyla projeye özgü bir Evrişimli Sinir Ağı (CNN) mimarisi tasarlanmıştır. Bu mimari, "Özellik Çıkarımı" ve "Sınıflandırma" olmak üzere iki temel veri işleme fazından oluşmaktadır. Gerçekleştirilen yapısal katman tercihleri ve bunların arka plandaki mühendislik gerekçeleri aşağıda detaylandırılmıştır:

1. Özellik Çıkarımı (Feature Extraction - Convolution & Pooling) Aşaması
Röntgen görüntülerindeki anatomik yapıların ve patolojik anomalilerin tespit edilebilmesi için 4 ardışık evrişim (Convolution) ve havuzlama (Pooling) bloğu kurgulanmıştır.

Evrişim (Conv2D) Katmanları ve Filtre Hiyerarşisi: Ağa giriş yapan görüntüler ilk aşamada 32 adet (3x3 boyutlu) filtre (kernel) ile taranarak akciğer sınırları ve kaburga kemikleri gibi temel, yüzeysel morfolojik hatlar tespit edilmiştir. Ağ derinleştikçe filtre sayısı kademeli olarak artırılmış (64, 128 ve 128) ve modelin yüzeysel çizgilerden ziyade, zatürreye (Pneumonia) özgü düzensiz beyaz sıvı birikintilerini (infiltrasyon/opaklık) ve karmaşık doku bozulmalarını saptaması hedeflenmiştir. Tüm evrişim işlemlerinde, ağa doğrusal olmayan (non-linear) bir hesaplama yeteneği kazandırmak ve negatif piksel değerlerini sıfırlayarak ağın yalnızca pozitif (anlamlı) sinyallere odaklanmasını sağlamak amacıyla ReLU (Rectified Linear Unit) aktivasyon fonksiyonu kullanılmıştır.

Havuzlama (MaxPooling2D) Katmanları: Her evrişim bloğunun ardına 2x2 boyutlu maksimum havuzlama (Max Pooling) katmanı eklenmiştir. Bu işlem, her 4 piksellik penceredeki en yüksek değerli (en baskın) özelliği alarak görüntülerin uzamsal (spatial) boyutlarını (224x224'ten kademeli olarak 12x12'ye kadar) küçültmüştür. Bu stratejik boyut indirgeme işlemi sayesinde; hesaplama maliyeti (RAM ve işlemci yükü) minimize edilmiş, modelin uzamsal kaymalara karşı direnci artırılmış ve resimdeki gereksiz piksellerin ezberlenmesinin (overfitting) önüne geçilerek en kritik patolojik izler damıtılmıştır.

In [3]:
# =====================================================================
# BÖLÜM 2: KARAR MERKEZİ (BEYİN - FLATTEN & DENSE)
# =====================================================================

# --- 5. BLOK: Düzleştirme (Flatten) ---
# Gözlerden gelen 14x14x128 boyutundaki 3 boyutlu küp şeklindeki özellik haritasını alıp,
# karar nöronlarının okuyabilmesi için upuzun tek boyutlu (1D) bir sayı şeridine dönüştürüyoruz.
model.add(Flatten())


# --- 6. BLOK: Gizli Katman (Mantık Nöronları) ---
# Düzleştirilmiş veriyi okuyup, aralarındaki bağlantılardan mantıksal "Zatürre Teşhisi"
# çıkarımı yapacak olan 256 adet yapay beyin hücresi (nöron) ekliyoruz.
model.add(Dense(units=256, activation='relu'))


# --- 7. BLOK: Ezberi Bozma (Dropout) ---
# Modelin eğitim resimlerini (gölgeleri, yazıları) körü körüne ezberlemesini (Overfitting) engellemek için,
# her eğitim adımında bu 256 nöronun yarısını (%50) rastgele uyutuyoruz/kapatıyoruz.
# Böylece model tek bir bölgeye saplanıp kalmak yerine resmin bütününe odaklanmak zorunda kalıyor.
model.add(Dropout(rate=0.5))


# --- 8. BLOK: Çıkış Katmanı (Son Karar Hakemi) ---
# İki sınıfımız (Zatürre=1, Sağlıklı=0) olduğu için sonuca sadece 1 adet hakem nöron koyuyoruz.
# 'sigmoid' aktivasyonu sayesinde bu nöron bize 0.0 ile 1.0 arasında matematiksel bir ihtimal veriyor.
# Örn: Çıktı 0.85 ise model %85 ihtimalle "Zatürre" teşhisi koyar.
model.add(Dense(units=1, activation='sigmoid'))


# Tüm kurduğumuz modelin yapısal özetini ve eğitilecek parametre sayılarını ekrana yazdırıyoruz:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 24, 24, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 12, 12, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 18432)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │     4,718,848 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,959,937 (18.92 MB)

 Trainable params: 4,959,937 (18.92 MB)

 Non-trainable params: 0 (0.00 B)

2. Karar Merkezi ve Sınıflandırma (Classification - Flatten & Dense) Aşaması
Evrişim katmanları tarafından çıkarılan çok boyutlu ve soyut özellik haritalarının, matematiksel ve mantıksal bir teşhis kararına dönüştürüldüğü nihai aşamadır.

Vektörizasyon (Flatten Katmanı): Özellik çıkarımı fazının sonunda elde edilen 3 boyutlu özellik küpleri, yapay sinir ağının (Dense) işleyebileceği veri yapısına uyarlanmak zorundadır. Bu doğrultuda Flatten katmanı kullanılarak, 3 boyutlu matrisler ezilmiş ve yaklaşık 18.000 elemanlı tek boyutlu (1D) kesintisiz bir sayı vektörüne dönüştürülmüştür.

Derin Bağlantılar ve Mantıksal Çıkarım (Dense Katmanı): Vektör haline getirilen özellikler, aralarındaki karmaşık ilişkileri ve çapraz bağlantıları kurmak üzere 256 nörondan (yapay beyin hücresi) oluşan tam bağlı (Fully Connected/Dense) bir gizli katmana aktarılmıştır. Bu katman, farklı bölgelerden gelen anomali sinyallerini birleştirerek "Zatürre" teşhisinin arka plandaki tıbbi mantığını matematiksel olarak kurgulamaktadır.

Regülarizasyon ve Ezberi Bozma (Dropout Katmanı): Yapay sinir ağlarının eğitim verilerindeki hastane logoları, tıbbi cihaz kabloları veya spesifik görüntü açıları gibi tesadüfi detayları ezberleme eğilimi bulunmaktadır. Bu aşırı öğrenme (Overfitting) problemini kesin olarak çözmek amacıyla modele %50 oranında (0.5) Dropout (seyreltme) uygulanmıştır. Bu yöntem, her eğitim iterasyonunda 256 nöronun yarısını rastgele inaktif duruma getirerek, ağın tek bir piksel grubuna bağımlı kalmasını engellemiş ve modelin resmi bir bütün olarak analiz edip genelleme yapabilmesini zorunlu kılmıştır.

Nihai Karar (Output Katmanı): Çözümlenmesi hedeflenen problem (Zatürre ve Normal) ikili (binary) bir sınıflandırma problemi olduğu için, modelin çıkış katmanında tek (1) bir karar nöronu konumlandırılmıştır. Bu nöronda 'Sigmoid' aktivasyon fonksiyonu kullanılarak, ağın ürettiği ham teşhis skorunun 0.0 ile 1.0 aralığında standart bir olasılık değerine (yüzdeye) dönüştürülmesi sağlanmıştır. Elde edilen değerin 0.5 eşiğinin üzerinde olması durumunda görüntünün pozitif (Zatürre), altında olması durumunda ise negatif (Normal) olarak etiketlenmesi matematiksel olarak güvence altına alınmıştır.

In [4]:
# =====================================================================
# BÖLÜM 3: ÖĞRENME KURALLARINI BELİRLEME (MODEL COMPILE)
# =====================================================================

# Modelin beynini (iskeletini) kurduk, şimdi nasıl öğreneceğini belirliyoruz.
model.compile(
    # 1. Optimizasyon Motoru (Optimizer): 
    # 'adam', modelin hata yaptığında ağırlıklarını (o 4.9 milyon sayıyı) nasıl düzelteceğini
    # hesaplayan çok zeki ve hızlı bir algoritmadır. Öğrenmenin beynidir.
    optimizer='adam',
    
    # 2. Kayıp Fonksiyonu (Loss Function):
    # Modelin yaptığı hataları cezalandırma sistemidir.
    # Sadece 2 sınıfımız (Zatürre=1, Normal=0) olduğu için 'binary_crossentropy' kullanıyoruz.
    loss='binary_crossentropy',
    
    # 3. Başarı Metriği:
    # Modelin eğitim sırasındaki performansını biz insanların anlayabileceği 
    # bir dilden (Doğruluk Oranı - % cinsinden) takip edebilmemizi sağlar.
    metrics=['accuracy']
)

print("Model başarıyla derlendi, motor takıldı ve eğitime hazır!")

Model başarıyla derlendi, motor takıldı ve eğitime hazır!


Bölüm 3: Öğrenme Kurallarının Belirlenmesi (Model Compile) - Tasarım Tercihlerim ve Gerekçeleri

Projemin bu aşamasında, Sequential yapısı ile uç uca ekleyerek fiziksel iskeletini tamamladığım evrişimli sinir ağı (CNN) modelimi eğitime hazır hale getiriyorum.

Keras mimarisinde model.compile kodunu kullanırken, ağın sonuna fiziksel yeni bir katman eklemiyorum. Aksine, inşası Dense(1) katmanıyla bitmiş olan o ölü tünelin içine; hataları ölçecek bir müfettiş (Loss), bu hatalardan ders çıkarıp sistemi güncelleyecek bir motor (Optimizer) ve süreci izleyebileceğim bir gösterge paneli (Metrics) entegre ediyorum.

Bu entegrasyonda yaptığım temel mühendislik tercihleri şunlardır:

Kayıp Fonksiyonu (loss = 'binary_crossentropy'): Teşhis problemim ikili doğaya (Zatürre=1, Normal=0) sahip olduğu için bu fonksiyonu seçtim. Modelimin ürettiği teşhis yüzdesi ile gerçek etiket arasındaki farkı matematiksel bir "ceza (hata) puanına" dönüştüren sistem budur.

Optimizasyon Motoru (optimizer = 'adam'): Modelim hata yaptığında, bu hatayı minimize etmek için geriye dönük hesaplamalar (backpropagation) yapması gerekiyor. Bu işlem için günümüzün endüstri standardı olan 'Adam' algoritmasını seçtim. Bu motor, modelimdeki o yaklaşık 4.9 milyon parametrenin (nöron bağının) her birini, hatayı en aza indirecek şekilde dinamik olarak güncelliyor.

Başarı Metriği (metrics = ['accuracy']): Proje yönergemizin "Değerlendirme Gereksinimleri (Bölüm 10)" kısmında benden Precision, Recall, F1-score ve Confusion Matrix gibi çok daha detaylı analizler istendiğinin farkındayım. Ancak bu compile aşamasında bilinçli olarak sadece Doğruluk (Accuracy) metriğini tanımladım. Bunun teknik sebebi; buradaki metriğin eğitim sürecinde (epoch'lar akarken) sadece anlık bir hız göstergesi olarak çalışmasıdır. Ağır ve karmaşık metrikleri eğitim döngüsünün içine sokmak, hesaplama maliyetini artırıp sistemi yavaşlatır. Eğitim sırasında kılavuzun "Bölüm 11" isterlerine uygun olarak modelin "overfitting" yapıp yapmadığını izlemek için Loss ve Accuracy grafiklerini takip etmem yeterlidir. Yönergelerde istenen o detaylı tıbbi metrikleri, modelimin eğitimi tamamen bittikten sonra, modelin daha önce hiç görmediği "Test Veri Seti" üzerinde ayrıntılı bir şekilde hesaplayıp raporlayacağım.